# Iterative Binding Optimization

**BioPipelines example** — iterative directed evolution loop that uses LigandMPNN to propose sequence variants and Boltz2 to evaluate their binding affinity, keeping only the best candidate each cycle.

[![Documentation](https://img.shields.io/badge/docs-readthedocs-blue)](https://biopipelines.readthedocs.io/en/latest/)
[![Preprint](https://img.shields.io/badge/preprint-bioRxiv-B31B1B)](https://www.biorxiv.org/content/10.64898/2026.03.11.711024v1)

In [ ]:
# Cell 1: Install BioPipelines and micromamba
!git clone https://github.com/locbp-uzh/biopipelines
%cd biopipelines
!pip install -e ".[all]"
!curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj -C /usr/local bin/micromamba
!micromamba create -f Environments/biopipelines.yaml -y

In [ ]:
# Cell 2: Install tools
from biopipelines.pipeline import *
from biopipelines.boltz2 import Boltz2
from biopipelines.ligand_mpnn import LigandMPNN
from biopipelines.mutation_profiler import MutationProfiler

with Pipeline(project="Setup", job="InstallTools"):
    Boltz2.install()
    LigandMPNN.install()
    MutationProfiler.install()

## Cell 3: Iterative Binding Optimization Pipeline

Starting from the NocT protein (PDB: 5OT9) and Histopine as ligand, this pipeline runs 5 cycles of:
1. Identify binding-pocket residues within 5 Å of the ligand
2. Generate 1000 sequence variants with LigandMPNN
3. Build a mutation frequency profile
4. Compose 3 candidate sequences by weighted random sampling (max 3 mutations)
5. Predict binding affinities with Boltz2 and keep the best candidate

In [ ]:
# Cell 3: Pipeline
from biopipelines.pipeline import *
from biopipelines.boltz2 import Boltz2
from biopipelines.ligand_mpnn import LigandMPNN
from biopipelines.distance_selector import DistanceSelector
from biopipelines.mutation_profiler import MutationProfiler
from biopipelines.mutation_composer import MutationComposer
from biopipelines.panda import Panda

with Pipeline(project="NocT", job="IterativeBindingOptimization"):
    protein = PDB("5OT9")
    ligand = Ligand("Histopine")
    original = Boltz2(proteins=protein,
                      ligands=ligand)

    current_best = Panda(tables=original.tables.affinity,
                         pool=original)

    for cycle in range(5):
        Suffix(f"Cycle{cycle+1}")
        pocket = DistanceSelector(structures=current_best,
                                  ligand="LIG",
                                  distance=5)
        variants = LigandMPNN(structures=current_best,
                              ligand="LIG",
                              num_sequences=1000,
                              redesigned=pocket.tables.selections.within)
        profile = MutationProfiler(original=current_best,
                                   mutants=variants)
        candidates = MutationComposer(frequencies=profile.tables.absolute_frequencies,
                                      num_sequences=3,
                                      mode="weighted_random",
                                      max_mutations=3)
        predicted = Boltz2(proteins=candidates,
                           ligands=ligand)
        current_best = Panda(tables=[current_best.tables.result,
                                     predicted.tables.affinity],
                             operations=[Panda.concat(add_source=True),
                                          Panda.sort("affinity_probability_binary",
                                                     ascending=False),
                                          Panda.head(1)],
                             pool=[current_best,
                                   predicted])